# 주행 데이터 YOLOv8 학습 (Google Colab)

`labeling_tool` 로 만든 `cap_image/` (프레임 + json 라벨) 을 YOLO 학습용 데이터셋으로 변환하고,
YOLOv8n 을 파인튜닝해서 `best.pt` 를 뽑아오는 노트북입니다.

- 클래스: `safety`(0), `danger`(1), `corn`(2)
- 입력: `cap_image/<영상폴더>/image/*.png|jpg` + `cap_image/<영상폴더>/json/*.json`
- 출력: `runs/safety_danger_corn/weights/best.pt` (+ 선택적으로 Hailo 변환용 ONNX)

---

## 실행 전 준비 (로컬 PC)

**1) GPU 런타임으로 바꾸기**
상단 메뉴 `런타임` → `런타임 유형 변경` → 하드웨어 가속기 **T4 GPU** → 저장.
(CPU로도 돌아가지만 100 epoch 기준 몇 시간 vs 몇 분 차이가 납니다.)

**2) `cap_image` 를 zip 으로 압축** — 프로젝트 폴더에서 PowerShell 로:

```powershell
cd C:\Users\curry\OneDrive\Desktop\metarobo\metarobo
Compress-Archive -Path cap_image -DestinationPath cap_image.zip -Force
```

**3) zip 을 Google Drive 에 업로드**
내 드라이브에 `metarobo` 폴더를 만들고 `cap_image.zip` 을 넣으면
경로가 `/content/drive/MyDrive/metarobo/cap_image.zip` 가 됩니다. (아래 3번 셀에서 경로 지정)

**4) 위에서부터 셀을 순서대로 실행** (`Shift + Enter`)

> Colab 런타임은 일정 시간 놀면 끊기고 `/content` 안의 파일이 전부 사라집니다.
> 학습이 끝나면 12번 셀에서 `best.pt` 를 **꼭 Drive 로 복사하거나 다운로드**하세요.

## 1. 환경 확인 (GPU 잡혔는지)

In [ ]:
!nvidia-smi

## 2. 패키지 설치

In [ ]:
!pip install -q ultralytics

import torch, ultralytics
ultralytics.checks()
print("torch:", torch.__version__, "| CUDA 사용 가능:", torch.cuda.is_available())

## 3. 설정

여기 값만 바꾸면 아래 셀들은 그대로 돌아갑니다.

- `DATA_SOURCE`
  - `"drive"` : Google Drive 에 올려둔 zip 사용 (권장)
  - `"upload"` : 브라우저에서 직접 파일 업로드 (수십 MB 이상이면 느림)
  - `"local"` : 이미 `/content/cap_image` 에 풀어놓은 경우

In [ ]:
# ── 데이터 ─────────────────────────────────────────────
DATA_SOURCE = "drive"                                         # "drive" | "upload" | "local"
ZIP_PATH    = "/content/drive/MyDrive/metarobo/cap_image.zip"  # DATA_SOURCE="drive" 일 때만 사용
CAP_IMAGE_DIR = "/content/cap_image"                           # 압축을 풀 위치
DATASET_DIR   = "/content/dataset"                             # 변환된 YOLO 데이터셋 위치

# ── 데이터셋 분할 ──────────────────────────────────────
VAL_RATIO = 0.2            # 검증 비율 (프레임 수 기준)
SEED = 42
INCLUDE_BACKGROUND = False # 박스가 0개인 프레임도 배경 샘플로 넣을지 (오탐이 많으면 True)

# ── 학습 ───────────────────────────────────────────────
EPOCHS = 100
IMGSZ  = 640
BATCH  = 16
MODEL_NAME  = "yolov8n.pt"   # 더 정확하게: yolov8s.pt / 가볍게: yolov8n.pt
RUN_NAME    = "safety_danger_corn"
PROJECT_DIR = "/content/runs"

CLASSES = ["safety", "danger", "corn"]  # class_id 0,1,2 — labeling_tool 과 순서가 같아야 함
print("설정 완료")

## 4. 데이터 가져오기 / 압축 풀기

Windows 에서 만든 zip 은 한글 파일명이 깨지는 경우가 있어서(CP949 인코딩),
이름을 직접 디코딩해가며 풀도록 했습니다. 역슬래시 경로와 `__MACOSX` 찌꺼기도 함께 처리합니다.

압축 안에 `cap_image/` 가 한 겹 더 감싸여 있든, 폴더 내용만 들어있든 상관없도록
**`<영상폴더>/json/*.json` 구조를 가진 폴더를 재귀적으로 찾아서** 사용합니다.
여기서 "라벨 폴더 0개" 가 나오면 아래에 실제 압축 해제 구조가 출력되니 그걸 보고 zip 을 다시 만드세요.

In [ ]:
import shutil, zipfile
from pathlib import Path

def unzip_safe(zip_path, dest):
    """한글(CP949) 파일명이 들어간 zip 도 깨지지 않게 푼다."""
    dest = Path(dest)
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    n = 0
    with zipfile.ZipFile(zip_path) as zf:
        for info in zf.infolist():
            name = info.filename
            if not info.flag_bits & 0x800:        # UTF-8 플래그가 없으면 CP949 로 추정
                try:
                    name = name.encode("cp437").decode("cp949")
                except (UnicodeEncodeError, UnicodeDecodeError):
                    pass
            name = name.replace("\\", "/")        # Windows 에서 만든 zip 의 역슬래시 경로 대응
            parts = [p for p in name.split("/") if p not in ("", ".", "..")]
            if not parts or parts[0] == "__MACOSX" or parts[-1] == ".DS_Store":
                continue
            target = dest.joinpath(*parts)
            if info.is_dir() or name.endswith("/"):
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(target, "wb") as out:
                shutil.copyfileobj(src, out)
            n += 1
    print(f"{n}개 파일 압축 해제 -> {dest}")


def find_video_dirs(base):
    """압축 구조가 어떻든 '<영상폴더>/json/*.json' 을 가진 폴더를 전부 찾아낸다.

    zip 이 cap_image/ 를 한 겹 더 감싸고 있든, 폴더 내용만 들어있든 상관없이 동작한다.
    """
    base = Path(base)
    return sorted({p.parent.parent for p in base.rglob("*.json") if p.parent.name == "json"})


def print_tree(base, max_depth=4, limit=40):
    """실제로 어떤 구조로 풀렸는지 확인용 출력."""
    base = Path(base)
    print(f"\n{base} 아래 실제 구조 (최대 {max_depth}단계):")
    shown = 0
    for p in sorted(base.rglob("*")):
        rel = p.relative_to(base)
        if len(rel.parts) > max_depth:
            continue
        if shown >= limit:
            print("  ... (생략)")
            break
        print(f"  {'[디렉터리]' if p.is_dir() else '[파일]    '} {rel}")
        shown += 1
    if shown == 0:
        print("  (비어 있음)")


if DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    assert Path(ZIP_PATH).exists(), f"{ZIP_PATH} 가 없습니다. Drive 안의 실제 경로를 확인하세요."
    unzip_safe(ZIP_PATH, CAP_IMAGE_DIR)

elif DATA_SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()          # cap_image.zip 선택
    unzip_safe(next(iter(uploaded)), CAP_IMAGE_DIR)

elif DATA_SOURCE == "local":
    assert Path(CAP_IMAGE_DIR).is_dir(), f"{CAP_IMAGE_DIR} 가 없습니다."

else:
    raise ValueError(f"알 수 없는 DATA_SOURCE: {DATA_SOURCE}")

# 라벨 폴더를 찾아서 이미지/라벨 개수 확인
video_dirs = find_video_dirs(CAP_IMAGE_DIR)
if video_dirs:
    print(f"\n라벨 폴더 {len(video_dirs)}개 발견")
    for d in video_dirs:
        n_img = len(list((d / "image").glob("*"))) if (d / "image").is_dir() else 0
        n_json = len(list((d / "json").glob("*.json")))
        flag = "" if n_img else "   <- image 폴더가 없습니다!"
        print(f"  {d.relative_to(CAP_IMAGE_DIR)}: 이미지 {n_img}장 / 라벨 {n_json}개{flag}")
else:
    print("\n[오류] '<영상폴더>/json/*.json' 구조를 찾지 못했습니다.")
    print("       zip 안에 json 폴더가 들어갔는지, 아래 실제 구조와 비교해보세요.")
    print_tree(CAP_IMAGE_DIR)

## 5. YOLO 데이터셋으로 변환

json 의 `bbox` 가 이미 YOLO 정규화 포맷(`x_center, y_center, width, height`, 0~1)이라
`class_id x_center y_center width height` 한 줄로 그대로 옮겨 적으면 됩니다.

분할에서 두 가지를 동시에 지킵니다.

1. **train/val 을 프레임이 아니라 "촬영 구간(clip)" 단위로 나눕니다.**
   같은 구간에서 뽑은 연속 프레임은 거의 똑같은 그림이라, 프레임 단위로 섞으면
   학습 데이터가 검증 데이터로 그대로 새어 들어가서(leakage) 실제보다 훨씬 좋은 mAP 가 나옵니다.
2. **클래스별로 val 비율을 맞춥니다.**
   지금처럼 폴더 하나에 한 클래스만 들어있는 구조(`cone/`, `danger/`, `safty/`)에서
   전체 프레임 수만 보고 나누면 val 이 한 클래스로만 채워져서, 나머지 클래스는
   성능 측정이 아예 안 됩니다. 그래서 클래스별 목표치에서 벗어난 정도를 계산해
   clip 을 하나씩 배정합니다.

한글/공백이 섞인 파일명은 ASCII 이름으로 바꿔 저장합니다(학습 중 경로 문제 방지).

In [ ]:
import json, random, re, shutil, unicodedata
from collections import Counter
from pathlib import Path

CAP = Path(CAP_IMAGE_DIR)
OUT = Path(DATASET_DIR)
IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".webp")
CLIP_PREFIX_RE = re.compile(r"^(\d{8}_\d{6})_")   # 예: 20260914_004845_001.json
MIN_SPLIT_CHUNK = 10                              # clip 을 이보다 작게는 쪼개지 않는다


def safe_stem(text, limit=40):
    """한글/공백 파일명 -> ASCII 안전 이름 (앞에 붙는 일련번호로 유일성 보장)."""
    text = unicodedata.normalize("NFKC", text)
    out = "".join(c if (c.isascii() and (c.isalnum() or c in "-_")) else "_" for c in text)
    return re.sub(r"_+", "_", out).strip("_")[:limit] or "img"


def resolve_image(image_dir, record, json_stem):
    """json 의 image 필드로 찾고, 없으면 json 과 같은 stem 의 이미지를 찾는다."""
    name = record.get("image")
    if name and (image_dir / name).exists():
        return image_dir / name
    for ext in IMAGE_EXTS:
        p = image_dir / f"{json_stem}{ext}"
        if p.exists():
            return p
    return None


def find_labeled_clips(video_dir):
    """{clip_key: [(image_path, label_lines), ...]}"""
    image_dir, json_dir = video_dir / "image", video_dir / "json"
    if not json_dir.is_dir():
        return {}
    clips, skipped = {}, 0
    for json_path in sorted(json_dir.glob("*.json")):
        record = json.loads(json_path.read_text(encoding="utf-8"))
        annotations = record.get("annotations", [])
        if not annotations and not INCLUDE_BACKGROUND:
            continue
        image_path = resolve_image(image_dir, record, json_path.stem)
        if image_path is None:
            skipped += 1
            continue
        lines = []
        for ann in annotations:
            b = ann["bbox"]
            lines.append(
                f"{int(ann['class_id'])} {b['x_center']:.6f} {b['y_center']:.6f} "
                f"{b['width']:.6f} {b['height']:.6f}"
            )
        m = CLIP_PREFIX_RE.match(json_path.stem)
        clip_key = f"{video_dir.name}::{m.group(1) if m else video_dir.name}"
        clips.setdefault(clip_key, []).append((image_path, lines))
    if skipped:
        print(f"  [주의] {video_dir.name}: 이미지를 못 찾은 라벨 {skipped}개 건너뜀")
    return clips


def frame_class(lines):
    """프레임 하나의 대표 클래스 = 그 프레임에서 박스가 가장 많은 클래스."""
    if not lines:
        return -1   # 배경 프레임
    return Counter(int(ln.split()[0]) for ln in lines).most_common(1)[0][0]


def clip_class_counts(frames):
    return Counter(frame_class(lines) for _, lines in frames)


def split_oversized_clips(per_clip, limit_of):
    """너무 큰 clip 은 촬영 순서를 유지한 채 절반씩 쪼갠다.

    clip 하나가 그 클래스의 val 목표치보다 크면, 그 clip 이 통째로 한쪽에 들어가면서
    반대쪽에 그 클래스가 아예 없어진다. 그래서 목표치보다 큰 clip 만 잘라준다.
    (순서를 유지한 채 앞/뒤로 자르므로 연속 프레임끼리는 계속 같이 붙어 다닌다.)
    """
    result, stack = {}, list(per_clip.items())
    while stack:
        key, frames = stack.pop()
        if len(frames) > limit_of(frames) and len(frames) >= MIN_SPLIT_CHUNK * 2:
            mid = len(frames) // 2
            stack.append((f"{key}#a", frames[:mid]))
            stack.append((f"{key}#b", frames[mid:]))
        else:
            result[key] = frames
    return result


# 1) 라벨 수집 (압축 구조에 상관없이 <영상폴더>/json 을 찾아서 사용)
video_dirs = find_video_dirs(CAP)
per_clip = {}
for d in video_dirs:
    per_clip.update(find_labeled_clips(d))

if not per_clip:
    # 어디서 막혔는지 구분해서 알려준다
    all_json = [p for p in CAP.rglob("*.json") if p.parent.name == "json"]
    n_with_box = 0
    for p in all_json:
        try:
            if json.loads(p.read_text(encoding="utf-8")).get("annotations"):
                n_with_box += 1
        except Exception:
            pass
    n_img = sum(1 for p in CAP.rglob("*") if p.suffix.lower() in IMAGE_EXTS)
    print_tree(CAP)
    raise SystemExit(
        f"\n라벨 폴더 {len(video_dirs)}개 / json {len(all_json)}개 "
        f"/ 박스가 있는 json {n_with_box}개 / 이미지 {n_img}장\n"
        "-------------------------------------------------------------\n"
        "- json 0개  : 압축에 json 폴더가 안 들어갔습니다. cap_image 폴더째로 다시 압축하세요.\n"
        "- 박스 0개  : 라벨링 결과가 비어 있습니다. labeling_tool 에서 박스를 그리고 S 로 저장하세요.\n"
        "              (배경 프레임도 학습에 쓰려면 3번 셀의 INCLUDE_BACKGROUND = True)\n"
        "- 이미지 0장: image 폴더가 빠졌습니다. 위 구조 출력에서 확인하세요.\n"
        "- 셋 다 정상이면 json 의 image 필드와 실제 파일명이 다른 경우입니다."
    )

total_frames = sum(len(v) for v in per_clip.values())

# 2) 클래스별 프레임 수를 세고, 그 클래스의 val 목표치보다 큰 clip 을 쪼갠다
class_frame_totals = Counter()
for frames in per_clip.values():
    class_frame_totals.update(clip_class_counts(frames))

smaller_share = min(VAL_RATIO, 1 - VAL_RATIO)

def limit_of(frames):
    counts = clip_class_counts(frames)
    dominant = counts.most_common(1)[0][0] if counts else -1
    base = class_frame_totals.get(dominant, total_frames)
    return max(MIN_SPLIT_CHUNK, round(base * smaller_share))

per_clip = split_oversized_clips(per_clip, limit_of)
clip_names = list(per_clip.keys())

# 3) 클래스별 목표치에서 벗어난 정도(오차)가 작아지는 쪽으로 clip 을 하나씩 배정
val_target = {c: n * VAL_RATIO for c, n in class_frame_totals.items()}
train_target = {c: n - n * VAL_RATIO for c, n in class_frame_totals.items()}

def deviation(val_counts, train_counts):
    """각 클래스가 목표 대비 얼마나 어긋났는지의 제곱합 (작을수록 좋음)."""
    score = 0.0
    for c in class_frame_totals:
        vt, tt = max(val_target[c], 1e-9), max(train_target[c], 1e-9)
        score += ((val_counts[c] - vt) / vt) ** 2 + ((train_counts[c] - tt) / tt) ** 2
    return score

rng = random.Random(SEED)
rng.shuffle(clip_names)
clip_names.sort(key=lambda n: len(per_clip[n]), reverse=True)   # 큰 clip 부터 배정

val_clips, train_clips = [], []
val_counts, train_counts = Counter(), Counter()
for name in clip_names:
    cc = clip_class_counts(per_clip[name])
    if deviation(val_counts + cc, train_counts) <= deviation(val_counts, train_counts + cc):
        val_clips.append(name); val_counts += cc
    else:
        train_clips.append(name); train_counts += cc

# 한쪽이 통째로 비면 반대쪽에서 가장 작은 clip 하나를 넘겨준다
if len(clip_names) >= 2 and (not val_clips or not train_clips):
    donor, receiver = (train_clips, val_clips) if not val_clips else (val_clips, train_clips)
    donor.sort(key=lambda n: len(per_clip[n]))
    receiver.append(donor.pop(0))

val_set = set(val_clips)

# 4) 파일 복사
if OUT.exists():
    shutil.rmtree(OUT)
for split in ("train", "val"):
    (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
    (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)

counts = {"train": 0, "val": 0}
for clip_idx, clip_name in enumerate(sorted(clip_names)):
    split = "val" if clip_name in val_set else "train"
    for frame_idx, (image_path, lines) in enumerate(per_clip[clip_name]):
        stem = f"c{clip_idx:02d}_{frame_idx:05d}_{safe_stem(image_path.stem)}"
        shutil.copy2(image_path, OUT / "images" / split / f"{stem}{image_path.suffix.lower()}")
        (OUT / "labels" / split / f"{stem}.txt").write_text(
            ("\n".join(lines) + "\n") if lines else "", encoding="utf-8")
        counts[split] += 1

# 5) data.yaml
data_yaml = OUT / "data.yaml"
data_yaml.write_text(
    f"path: {OUT}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"nc: {len(CLASSES)}\n"
    f"names: {CLASSES}\n",
    encoding="utf-8",
)

print(f"\n전체 라벨 프레임 {total_frames}장 / 영상 폴더 {len(video_dirs)}개 / 구간(clip) {len(clip_names)}개")
print(f"train: {counts['train']}장  (clip {len(train_clips)}개)")
print(f"val  : {counts['val']}장  (clip {len(val_clips)}개)")
print(f"\n클래스별 분할 (목표 val 비율 {VAL_RATIO:.0%})")
for c in sorted(class_frame_totals):
    name = CLASSES[c] if 0 <= c < len(CLASSES) else "background"
    v, t = val_counts[c], train_counts[c]
    print(f"  {name:<11} train {t:>4} / val {v:>4}  (val {v / (v + t):.0%})" if v + t else f"  {name}: 0장")
print(f"\ndata.yaml:\n{data_yaml.read_text(encoding='utf-8')}")

## 6. 데이터셋 점검

학습을 돌리기 전에 여기서 한 번 거릅니다. 특히 **val 에 없는 클래스**가 있으면
그 클래스의 mAP 는 의미가 없으니, 라벨을 더 만들거나 `SEED` / `VAL_RATIO` 를 바꿔보세요.

In [ ]:
from collections import Counter
from pathlib import Path

OUT = Path(DATASET_DIR)
problems = []

for split in ("train", "val"):
    label_dir = OUT / "labels" / split
    img_dir = OUT / "images" / split
    counter, n_boxes, n_empty = Counter(), 0, 0
    for txt in label_dir.glob("*.txt"):
        rows = [r for r in txt.read_text(encoding="utf-8").splitlines() if r.strip()]
        if not rows:
            n_empty += 1
        for r in rows:
            parts = r.split()
            cid, vals = int(parts[0]), [float(v) for v in parts[1:]]
            counter[cid] += 1
            n_boxes += 1
            if not 0 <= cid < len(CLASSES):
                problems.append(f"{txt.name}: 잘못된 class_id {cid}")
            if any(v < 0 or v > 1 for v in vals):
                problems.append(f"{txt.name}: 0~1 범위를 벗어난 좌표 {vals}")
            elif vals[2] <= 0 or vals[3] <= 0:
                problems.append(f"{txt.name}: 너비/높이가 0 이하 {vals}")

    n_img = len([p for p in img_dir.iterdir() if p.is_file()])
    print(f"[{split}] 이미지 {n_img}장 / 박스 {n_boxes}개 / 박스 없는 프레임 {n_empty}장")
    for i, name in enumerate(CLASSES):
        print(f"    {i} {name:<7}: {counter[i]:>5}개")
    if split == "val":
        missing = [CLASSES[i] for i in range(len(CLASSES)) if counter[i] == 0]
        if missing:
            problems.append(f"val 에 없는 클래스: {missing} -> 이 클래스는 성능 측정이 안 됩니다")

print()
print("\n".join(f"[주의] {p}" for p in problems[:20]) if problems else "문제 없음 ✅")

## 7. 라벨 눈으로 확인 (가장 흔한 사고: 클래스 순서가 뒤바뀐 경우)

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

OUT = Path(DATASET_DIR)
COLORS = ["#22c55e", "#ef4444", "#f59e0b"]   # safety / danger / corn

samples = sorted((OUT / "images" / "train").iterdir())
samples = random.Random(0).sample(samples, min(6, len(samples)))

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, img_path in zip(axes.ravel(), samples):
    img = Image.open(img_path)
    W, H = img.size
    ax.imshow(img); ax.axis("off"); ax.set_title(img_path.name[:30], fontsize=8)
    label = OUT / "labels" / "train" / f"{img_path.stem}.txt"
    for row in label.read_text(encoding="utf-8").splitlines():
        if not row.strip():
            continue
        p = row.split()
        cid = int(p[0]); xc, yc, w, h = map(float, p[1:5])
        x, y = (xc - w / 2) * W, (yc - h / 2) * H
        ax.add_patch(patches.Rectangle((x, y), w * W, h * H, lw=2,
                                       edgecolor=COLORS[cid % len(COLORS)], facecolor="none"))
        ax.text(x, max(y - 4, 0), CLASSES[cid], color="white", fontsize=9,
                bbox=dict(facecolor=COLORS[cid % len(COLORS)], edgecolor="none", pad=1))
for ax in axes.ravel()[len(samples):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 8. 학습

처음에는 `EPOCHS = 30` 정도로 짧게 돌려 파이프라인이 제대로 도는지 확인하고,
문제 없으면 100~200 으로 올려 본 학습을 하는 걸 추천합니다.

- `patience` : 지정한 epoch 동안 val 성능이 안 오르면 조기 종료
- `fliplr=0.5` : 좌우 반전 증강. 표지판에 글자/방향이 들어가면 `0.0` 으로 낮추세요.

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

device = 0 if torch.cuda.is_available() else "cpu"
print("device:", device)

model = YOLO(MODEL_NAME)
results = model.train(
    data=str(Path(DATASET_DIR) / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=device,
    project=PROJECT_DIR,
    name=RUN_NAME,
    exist_ok=True,
    seed=SEED,
    patience=30,
    workers=2,
    plots=True,
    # ── 증강 (데이터가 적을수록 조금 더 세게) ──
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.4,
    fliplr=0.5, mosaic=1.0,
)

SAVE_DIR = Path(results.save_dir)
BEST_PT = SAVE_DIR / "weights" / "best.pt"
print("\n학습 완료:", SAVE_DIR)
print("best.pt:", BEST_PT, "| 존재:", BEST_PT.exists())

## 9. 학습 결과 확인

- `results.png` : loss / precision / recall / mAP 곡선.
  train loss 는 떨어지는데 val loss 가 올라가면 과적합 → epoch 을 줄이거나 데이터를 늘리세요.
- `confusion_matrix_normalized.png` : 어떤 클래스를 어떤 클래스로 헷갈리는지.

In [ ]:
from IPython.display import Image as IPyImage, display
from pathlib import Path

for fname in ["results.png", "confusion_matrix_normalized.png",
              "val_batch0_labels.jpg", "val_batch0_pred.jpg"]:
    p = Path(SAVE_DIR) / fname
    if p.exists():
        print("──", fname)
        display(IPyImage(filename=str(p), width=900))

## 10. 검증 지표 (클래스별 mAP)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best = YOLO(str(BEST_PT))
metrics = best.val(data=str(Path(DATASET_DIR) / "data.yaml"), imgsz=IMGSZ, device=device)

print(f"\n전체 mAP50    : {metrics.box.map50:.3f}")
print(f"전체 mAP50-95 : {metrics.box.map:.3f}\n")
ap_index = list(metrics.ap_class_index)
for i, name in enumerate(CLASSES):
    if i in ap_index:
        j = ap_index.index(i)
        print(f"  {name:<7} P={metrics.box.p[j]:.3f}  R={metrics.box.r[j]:.3f}  "
              f"mAP50={metrics.box.ap50[j]:.3f}  mAP50-95={metrics.box.maps[i]:.3f}")
    else:
        print(f"  {name:<7} (val 에 샘플 없음)")

## 11. 검증 이미지로 실제 추론해보기

In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path

val_images = sorted((Path(DATASET_DIR) / "images" / "val").iterdir())
picks = random.Random(1).sample(val_images, min(6, len(val_images)))

preds = best.predict(source=[str(p) for p in picks], conf=0.4, imgsz=IMGSZ,
                     device=device, verbose=False)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, r in zip(axes.ravel(), preds):
    ax.imshow(r.plot()[:, :, ::-1])   # BGR -> RGB
    ax.axis("off")
    ax.set_title(f"{len(r.boxes)} detections", fontsize=9)
for ax in axes.ravel()[len(preds):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

## 12. 결과 저장 (중요)

Colab 런타임이 끊기면 `/content` 는 전부 날아갑니다. 둘 중 하나는 꼭 하세요.
저장한 `best.pt` 는 로컬 저장소의 `yolo_learning/best.pt` 자리에 덮어쓰면 됩니다.

In [ ]:
# ① Google Drive 로 복사 (Drive 를 마운트한 경우)
import shutil
from pathlib import Path

DRIVE_OUT = Path("/content/drive/MyDrive/metarobo")
if Path("/content/drive/MyDrive").exists():
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BEST_PT, DRIVE_OUT / "best.pt")
    shutil.make_archive(str(DRIVE_OUT / f"run_{RUN_NAME}"), "zip", SAVE_DIR)  # 학습 로그/그래프 전체
    print("Drive 저장 완료:", DRIVE_OUT / "best.pt")
else:
    print("Drive 가 마운트되어 있지 않습니다. 아래 ② 로 다운로드하세요.")

In [ ]:
# ② 브라우저로 바로 다운로드
from google.colab import files
files.download(str(BEST_PT))

## 13. (선택) Hailo 변환용 ONNX 내보내기

라즈베리파이 + Hailo NPU 로 올릴 거라면 `.pt` 를 ONNX 로 먼저 뽑아야 합니다.
이후 Hailo Dataflow Compiler 로 `.hef` 변환은 로컬에서 진행하세요.

In [ ]:
onnx_path = best.export(format="onnx", imgsz=IMGSZ, opset=11, simplify=True)
print("ONNX:", onnx_path)

from google.colab import files
files.download(str(onnx_path))

---

## 결과가 잘 안 나올 때

| 증상 | 확인할 것 |
|---|---|
| mAP 가 0 에 가까움 | 7번 셀에서 박스가 물체와 맞는지 — 클래스 순서(`safety, danger, corn`)가 라벨링 툴과 같은지 |
| val mAP 만 유난히 높음(0.99 등) | train/val 이 같은 구간에서 나왔을 가능성. 5번 셀의 clip 개수가 2개 이상인지 확인 |
| 특정 클래스만 성능이 낮음 | 6번 셀의 클래스별 박스 개수 — 수가 적으면 그 클래스 프레임을 더 라벨링 |
| 학습 중 CUDA out of memory | `BATCH` 를 8 또는 4 로, 그래도 안 되면 `IMGSZ` 를 416 으로 |
| 실제 주행 영상에서 오탐이 많음 | `INCLUDE_BACKGROUND = True` 로 배경(객체 없음) 프레임을 학습에 포함 |